In [34]:
import numpy as np
L=4
dim_k = 8
dim_v= 8
k = np.random.randn(L, dim_k)
q = np.random.randn(L, dim_k)
v= np.random.randn(L, dim_v)

In [35]:

print("Q\n", q)
print("K\n", k)
print("V\n", v)

Q
 [[-1.91890378  0.04205626 -0.53400282  0.89083672  0.49902124  1.797466
   1.26385825 -0.8834793 ]
 [ 1.14045078 -1.5699704   2.00879325 -0.12504634 -0.95051504 -0.38050151
  -1.97531986  1.0782272 ]
 [-1.3063994  -0.06609734 -0.20704792 -1.22578697  1.03757925 -0.14663006
   0.09481282  0.65426909]
 [-0.75103901  0.91995137  1.26145082  2.52416083 -0.61157039 -1.11296936
   0.5148003  -0.38861273]]
K
 [[ 0.46020393 -0.8133392  -1.09248986  0.64293035 -0.78328164 -0.59786999
  -0.89755799 -1.22066548]
 [-0.77417776 -0.74143982  1.83675342  0.69233855 -1.53720894  0.22906965
  -1.41077521  2.15179325]
 [ 0.6203868   1.72184675 -1.05803614 -0.73433226  2.10040676 -0.7000176
  -1.50466274  0.46484243]
 [ 0.21112413  0.30639926  0.60544309  1.27421954 -0.65009132  0.90987061
  -0.05247522  1.68637001]]
V
 [[ 0.4981105   1.68458183 -0.09573878  2.2753387   0.74248003 -0.25367817
  -1.3983386  -0.92740712]
 [ 0.19688346  0.19508622  0.67797086 -0.50046472 -0.11889555 -0.72687379
   0.1726

self attention = softmax= (q.k^t/d^1/2   + Mask )------
new v = selfattention.v

In [36]:
np.matmul(q,k.T)

array([[-1.28263297, -2.94912013, -3.72969334,  0.17443023],
       [ 0.95559688, 10.36504358, -2.28599262,  3.01027222],
       [-2.71814324, -0.52303535,  2.6383695 , -1.69291325],
       [ 0.30761961,  3.08657932, -3.53084568,  2.80594109]])

In [37]:
scaled = np.matmul(q,k.T)/np.sqrt(dim_k)
print(scaled)

[[-0.45347924 -1.04267142 -1.31864573  0.0616704 ]
 [ 0.33785452  3.6645963  -0.80822044  1.06429195]
 [-0.96100876 -0.18492092  0.93280448 -0.59853522]
 [ 0.10875996  1.09127058 -1.24834246  0.99204999]]


Mask

lower triangular matrix transformed to  a combination of 0 and inf. where 0 keeps the scaled values unaffected and inf makes it so that those values are ruined(hidden)

This is to ensure words don't get context from words generated in the future.
Not required in the encoders, but required int he decoders

In [38]:
mask=np.tril(np.ones((L,L)))
print(m)

[[1. 0. 0. 0.]
 [1. 1. 0. 0.]
 [1. 1. 1. 0.]
 [1. 1. 1. 1.]]


In [39]:
mask[mask==0]=-np.inf
mask[mask==1]=0

print(mask)

[[  0. -inf -inf -inf]
 [  0.   0. -inf -inf]
 [  0.   0.   0. -inf]
 [  0.   0.   0.   0.]]


In [40]:
scaled + mask

array([[-0.45347924,        -inf,        -inf,        -inf],
       [ 0.33785452,  3.6645963 ,        -inf,        -inf],
       [-0.96100876, -0.18492092,  0.93280448,        -inf],
       [ 0.10875996,  1.09127058, -1.24834246,  0.99204999]])

Softmax

In [41]:
def softmax(x):
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

In [42]:
attention = softmax(scaled + mask)

In [43]:
attention

array([[1.        , 0.        , 0.        , 0.        ],
       [0.0346651 , 0.9653349 , 0.        , 0.        ],
       [0.10185776, 0.22133228, 0.67680996, 0.        ],
       [0.15754471, 0.42082621, 0.04055288, 0.3810762 ]])

In [44]:
new_v= np.matmul(attention,v)
print(new_v)

[[ 0.4981105   1.68458183 -0.09573878  2.2753387   0.74248003 -0.25367817
  -1.3983386  -0.92740712]
 [ 0.20732553  0.24671972  0.65115014 -0.40424122 -0.08903588 -0.71047042
   0.1182125   0.46630991]
 [ 1.03322023 -0.524399    0.38561576 -0.98042786  0.58126085 -0.32244429
   0.05346667 -0.51364086]
 [ 0.33672519  0.84801348  0.79929779 -0.35957971 -0.17784151 -0.39221639
  -0.4566626  -0.10142539]]


In [45]:
print("previous v: ", v)


previous v:  [[ 0.4981105   1.68458183 -0.09573878  2.2753387   0.74248003 -0.25367817
  -1.3983386  -0.92740712]
 [ 0.19688346  0.19508622  0.67797086 -0.50046472 -0.11889555 -0.72687379
   0.17267172  0.51635818]
 [ 1.38725373 -1.09213169  0.36245131 -1.6273693   0.78596498 -0.20053571
   0.23297599 -0.7882035 ]
 [ 0.31263999  1.42965692  1.34979456 -1.15841428 -0.72598105 -0.10032349
  -0.83572363 -0.36908742]]


The whole self attention function:

In [46]:
def softmax(x):
  return (np.exp(x).T / np.sum(np.exp(x), axis=-1)).T

def scaled_dot_product_attention(q, k, v, mask=None):
  d_k = q.shape[-1]
  scaled = np.matmul(q, k.T) / np.sqrt(dim_k)
  if mask is not None:
    scaled = scaled + mask
  attention = softmax(scaled)
  out = np.matmul(attention, v)
  return out, attention

In [47]:
mask=np.tril(np.ones((L,L)))
mask[mask==0]=-np.inf
mask[mask==1]=0


In [48]:
values, attention = scaled_dot_product_attention(q, k, v, mask=mask)
print("Q\n", q)
print("K\n", k)
print("V\n", v)
print("New V\n", values)
print("Attention\n", attention)

Q
 [[-1.91890378  0.04205626 -0.53400282  0.89083672  0.49902124  1.797466
   1.26385825 -0.8834793 ]
 [ 1.14045078 -1.5699704   2.00879325 -0.12504634 -0.95051504 -0.38050151
  -1.97531986  1.0782272 ]
 [-1.3063994  -0.06609734 -0.20704792 -1.22578697  1.03757925 -0.14663006
   0.09481282  0.65426909]
 [-0.75103901  0.91995137  1.26145082  2.52416083 -0.61157039 -1.11296936
   0.5148003  -0.38861273]]
K
 [[ 0.46020393 -0.8133392  -1.09248986  0.64293035 -0.78328164 -0.59786999
  -0.89755799 -1.22066548]
 [-0.77417776 -0.74143982  1.83675342  0.69233855 -1.53720894  0.22906965
  -1.41077521  2.15179325]
 [ 0.6203868   1.72184675 -1.05803614 -0.73433226  2.10040676 -0.7000176
  -1.50466274  0.46484243]
 [ 0.21112413  0.30639926  0.60544309  1.27421954 -0.65009132  0.90987061
  -0.05247522  1.68637001]]
V
 [[ 0.4981105   1.68458183 -0.09573878  2.2753387   0.74248003 -0.25367817
  -1.3983386  -0.92740712]
 [ 0.19688346  0.19508622  0.67797086 -0.50046472 -0.11889555 -0.72687379
   0.1726